<h1>Lecture9 : A Simple Linear Regression Model with brms</h1><h2>Instructor: Dr. Hu Chuan-Peng</h2><p></p>

<h2>序言</h2><blockquote><p>⭐ 在之前的课程中，我们学习了贝叶斯统计方法和使用代码实现统计模型。</p><p>然而，我们并没有使用贝叶斯模型深入到真正的心理学研究当中。</p><p>从本节课开始，我们将通过一个个真实的案例，带领大家进入心理学贝叶斯统计分析的世界。</p></blockquote><p></p>

## 研究示例： 自我加工优势 (Self-prioritization Effect, SPE)  

在本节课，我们关注的研究问题是 “自我作为人类独特的心理概念，是否在也会促进认知加工的表现？”。  

特别地，我们关注的是自我在知觉匹配任务中的作用，**探究自我和他人条件下，人们的认知加工差异，尤其是在反应时间上的表现。**  

> 探究自我加工的优势通畅使用匹配范式任务中“自我（self）”和“他人（other）”的认知匹配之间的关系 (Sui et al., 2012)。  
> * 在自我匹配任务中，被试首先学习几何图形和身份标签的关系。例如，三角形代表自我；圆形代表他人。在随后的知觉匹配判断任务中，需要判断所呈现的几何图形和文字标签是否与之前学习的关系相匹配。  
> * 想象一下你在完成自我匹配任务时，面对不同的刺激：有时你可能会觉得某个“自我”相关的图像比“他人”相关的更具吸引力，或许这反映了你对自己本身具有更多的关注。  

![Image Name](https://cdn.kesci.com/upload/smipfxtgj4.png?imageView2/0/w/640/h/640)  

> Sui, J., He, X., & Humphreys, G. W. (2012). Perceptual effects of social salience: Evidence from self-prioritization effects on perceptual matching. Journal of Experimental Psychology: Human Perception and Performance, 38(5), 1105–1117. https://doi.org/10.1037/a0029792  


根据Sui et al., （2012）的文章，我们假设，**在“自我”条件下，个体的反应时间会快于在“他人”条件下的反应时间。** 那么在贝叶斯的框架下，我们应该如何解决我们的研究问题以及验证我们的研究假设是否正确呢？  

![Image Name](https://cdn.kesci.com/upload/smkhdwv5zt.png?imageView2/0/w/720)  

在本节课中，我们将学习如何使用大家所熟悉的简单线性模型 (linear regression model) 来检验心理学效应。  

包括以下内容：  

1. **简单线性模型**。  
2. **先验预测检验**。  
3. 模型拟合和诊断。  
4.  后验推理。  
5. **模型检验 (后验预测检验)**。  

<div style="padding-bottom: 20px;"></div>

<p>我们使用的数据来自于Kolvoort等人（2020），该数据集包含了多个被试在自我匹配范式下的行为数据。数据集涉及了不同年龄、性别、文化背景的健康成年被试。</p><ul><li><p>我们使用  <code>read_csv</code> 方法来读取数据 <code>Kolvoort_2020_HBM_Exp1_Clean.csv</code> (数据已经预先存放在和鲸平台中)。</p></li><li><p>数据包含多个变量，选择我们需要的<code>Label</code> 表示标签（self / other），<code>RT_sec</code> 表示被试的反应时间。</p></li><li><p>每一行(index)表示一个trial数。</p></li></ul><blockquote><ul><li><p>数据来源: Kolvoort, I. R., Wainio‐Theberge, S., Wolff, A., &amp; Northoff, G. (2020). Temporal integration as “common currency” of brain and self‐scale‐free activity in resting‐state EEG correlates with temporal delay effects on self‐relatedness. Human brain mapping, 41(15), 4355-4374.</p></li></ul></blockquote><p></p>

In [1]:
# 导入所需库
options(repos = c(CRAN = "https://mirrors.tuna.tsinghua.edu.cn/CRAN/"))
if (!requireNamespace('pacman', quietly = TRUE)) {
    install.packages('pacman')
}
pacman::p_load("tidyverse","ggplot2", "dplyr","gridExtra","papaja", "patchwork","bayesplot","rstan","brms") # "bayesrules"

In [2]:
# 加载数据
tryCatch({
  df_raw <- read_csv("/home/mw/input/bayes3797/Kolvoort_2020_HBM_Exp1_Clean.csv")
}, error = function(e) {
  df_raw <- read_csv("2024/data/Kolvoort_2020_HBM_Exp1_Clean.csv")
})

# 显示数据前几行
head(df_raw)

New names:
• `` -> `...1`
Rows: 10018 Columns: 15
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (6): Handedness, First_Language, Education, Countryself, Countryparents,...
dbl (9): ...1, Subject, Age, Shape, Label, Response, RT_ms, RT_sec, ACC

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


...1,Subject,Age,Handedness,First_Language,Education,Countryself,Countryparents,Shape,Label,Matching,Response,RT_ms,RT_sec,ACC
<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,201,18,r,English/Farsi,High School,Iran/Canada,Iran,3,2,Matching,1,753,0.753,1
2,201,18,r,English/Farsi,High School,Iran/Canada,Iran,3,2,Matching,1,818,0.818,1
3,201,18,r,English/Farsi,High School,Iran/Canada,Iran,1,3,Matching,1,917,0.917,1
4,201,18,r,English/Farsi,High School,Iran/Canada,Iran,3,2,Matching,1,717,0.717,1
5,201,18,r,English/Farsi,High School,Iran/Canada,Iran,3,2,Matching,1,988,0.988,1
6,201,18,r,English/Farsi,High School,Iran/Canada,Iran,3,2,Matching,1,950,0.950,1


In [3]:
# 筛选出被试"201"，匹配类型为"Matching"的数据
df_raw$Subject <- as.character(df_raw$Subject)
df <- df_raw %>%
  filter(Subject == "201" & Matching == "Matching") %>%
  # 选择需要的两列
  select(Label, RT_sec) %>%
  # 重新编码标签（Label）
  mutate(Label = case_when(
    Label == 1 ~ 0,
    Label == 2 ~ 1,
    Label == 3 ~ 1
  )) %>%
  # 设置索引
  mutate(index = row_number()) %>%
  column_to_rownames("index")

# 显示处理后的数据前几行
head(df)

,Label,RT_sec
,<dbl>,<dbl>
1,1,0.753
2,1,0.818
3,1,0.917
4,1,0.717
5,1,0.988
6,1,0.950


<p>进一步可视化数据情况</p><ul><li><p>我们使用 <code>gglot</code> 方法来进行可视化。</p></li><li><p>其中横轴为Label <code>x="Label"</code>，纵轴为反应时间 <code>y="RT_sec"</code>。</p></li></ul><p></p>

In [4]:
# 计算每个Label条件下的均值
mean_values <- df %>%
  group_by(Label) %>%
  summarise(mean_RT = mean(RT_sec), .groups = "drop")

# 绘制符合APA格式的箱线图
ggplot(df, aes(x = factor(Label), y = RT_sec)) +
  geom_boxplot() +
  geom_line(data = mean_values, aes(x = factor(Label), y = mean_RT, group = 1), 
            color = "red", linewidth = 1) +
  geom_point(data = mean_values, aes(x = factor(Label), y = mean_RT), 
             color = "red", size = 3) +
  # 使用APA格式主题
  theme_apa() +
  # 添加标签（APA格式通常要求清晰简洁的标签）
  labs(x = "Label Condition (0 = self, 1 = other)",
       y = "Reaction Time (sec)") +
  # 调整图形大小（APA建议图形比例协调）
  theme(plot.width = unit(5, "in"),
        plot.height = unit(3.2, "in"))


Warning message in plot_theme(plot):
“The `plot.width` theme element is not defined in the element hierarchy.”
Warning message in plot_theme(plot):
“The `plot.height` theme element is not defined in the element hierarchy.”


plot without title

## 简单线性回归：使用线性模型表示两种条件下反应的差异  

### 频率学派视角下的回归模型  

让我们先回顾在传统频率学派的视角下，回归模型的建立和检验一般基于参数估计和假设检验：  

1. **构建模型**：我们可以使用一个简单线性回归模型，其中反应时间（RT_sec）作为因变量，自变量为二分离散变量（Label）。我们可以将 `self` 编码为 0，`other` 编码为 1，这样模型将估计出“自我”条件相较于“他人”条件在反应时间上的效应。  

模型形式为：  

$$  
   RT_{sec} = \beta_0 + \beta_1 \cdot Label + \epsilon  
$$  

其中，$\beta_0$ 表示“self”条件下的平均反应时间，$\beta_1$ 表示other条件下相较于self条件的反应时间差异。  

2. **假设检验**：在该模型中，$\beta_1$ 的显著性可以用 t 检验来判断。如果 $\beta_1$ 显著不为 0（例如 $p < 0.05$），则说明自我条件下的反应时间显著不同于他人条件，即存在自我加工的优势。  

3. **模型解释**：若 $\beta_1$ 为负值，则表明自我条件的反应时间较短，暗示自我加工速度较快。


### 贝叶斯视角下的回归模型  

在贝叶斯视角下，回归模型的建立和检验不同于传统的假设检验，而是通过对参数的后验分布进行推断：  

1. **构建贝叶斯模型**：贝叶斯模型和频率学派的回归模型形式相同，但其参数估计基于贝叶斯推断。我们会为 $\beta_0$ 和 $\beta_1$ 指定先验分布（例如，正态分布），并结合观测数据计算其后验分布：  

$$  
RT_{sec} \sim \mathcal{N}(\beta_0 + \beta_1 \cdot Label, \sigma^2)  
$$  

1. **计算后验分布**：使用贝叶斯推断方法（如 MCMC 采样）得到 $\beta_1$ 的后验分布。  

2. **显著性检验**：通过后验分布检验 $\beta_1$ 是否显著，例如计算 $\beta_1 > 0$ 或 $\beta_1 < 0$ 的概率，或计算最高密度区间（HDI）。如果 95% HDI 不包含 0，可以认为自我条件和他人条件在反应时间上的差异是显著的。  

3. **模型解释**：在贝叶斯框架下，我们不仅可以观察参数的点估计（如 $\beta_1$ 的均值），还可以通过后验分布和 HDI 提供更加直观的置信水平解释。

**⭐贝叶斯回归模型的可视化表达**  

*  <span style = "color: orange;">预测值</span> $\mu_i$(即直线上橙色的点)可以写为：$\mu_i = \beta_0 + \beta_1 X_i(1)$  

* 从图上可以看到<span style = "color: orange;">预测值</span>和<span style = "color: gray;">实际值</span> (即灰色的散点)之间存在出入，实际值会在预测值附近波动  

* 那么实际值可以看作服从以$\mu_i$为均值，标准差为$\sigma$的正态分布，即：$Y_i \sim N(\mu_i, \sigma ^ 2)$  

![Image Name](https://cdn.kesci.com/upload/smkijcu8co.png?imageView2/0/w/700)  

*(改编自：https: // saylordotorg.github.io/text_introductory-statistics/s14-03-modelling-linear-relationships.html)*  

<br><br>  

<div style="padding-bottom: 20px;"></div>

**贝叶斯回归模型的数学表达式**  


$$  
\begin{align*}  
\beta_0   &\sim N\left(m_0, s_0^2 \right)  \\  
\beta_1   &\sim N\left(m_1, s_1^2 \right)  \\  
\sigma    &\sim \text{Exp}(\lambda)        \\  
&\Downarrow \\  
\mu_i &= \beta_0 + \beta_1 X_i + \sigma      \\  
&\Downarrow \\  
Y_i | \beta_0, \beta_1, \sigma &\stackrel{ind}{\sim} N\left(\mu_i, \sigma^2\right). \\  
\end{align*}  
$$  

* 回归模型需满足如下假设：  

    1. 独立观测假设:每个观测值$Y_i$是相互独立的，即一个被试的反应时间不受其他被试的影响。  

    2. 线性关系假设: 预测值$\mu_i$和自变量$X_i$之间可以用线性关系来描述，即：$\mu_i = \beta_0 + \beta_1 X_i$  

    3. 方差同质性假设： 在任意自变量的取值下，观测值$Y_i$都会以$\mu_i$为中心，同样的标准差$\sigma$呈正态分布变化（$\sigma$ is consistent）  





## 定义先验  

在贝叶斯的分析框架中，我们需要为模型中的每个参数设置先验分布。  

而根据之前的模型公式(数据模型)可发现，我们的$Y$为被试反应时间(RT_sec)，$X$为标签（Label），并且存在三个未知的参数$\beta_0，\beta_1，\sigma$ 。  

因此，我们需要对每个未知的参数定义先验分布。  

$$  
\beta_0    \sim N\left(m_0, s_0^2 \right)  \\  
\beta_1   \sim N\left(m_1, s_1^2 \right) \\  
\sigma \sim \text{Exp}(\lambda)  
$$  


> * 参数的前提假设(assumptions):  
>    * $\beta_0，\beta_1，\sigma$ 之间相互独立  
> * 此外，规定 $\sigma$ 服从指数分布，以限定其值恒为正数  
> * 其中，$m_0，s_0，m_1，s_1$为超参数  
>    * 我们需要根据我们对$\beta_0$和$\beta_1$的先验理解来选择超参数的范围  
>    * 比如，$\beta_1$反映了标签从 self 切换到 other 时，反应时间的平均变化值；$\beta_0$反映了在 other 条件下的基础反应时间  


**指定超参数**  

$$  
\begin{equation}  
\begin{array}{lcrl}  
\text{data:} & \hspace{.05in} &   Y_i | \beta_0, \beta_1, \sigma & \stackrel{ind}{\sim} N\left(\mu_i, \sigma^2\right) \;\; \text{ with } \;\; \mu_i = \beta_0 + \beta_1X_i \\  

\text{priors:} & & \beta_{0}  & \sim N\left(5, 2^2 \right)  \\  
                    & & \beta_1  & \sim N\left(0, 1^2 \right) \\  
                    & & \sigma   & \sim \text{Exp}(0.3)  \\  
\end{array}  
\end{equation}  
$$  

这里，我们根据生活经验或直觉对超参数进行了定义：  

* 其次，我们假设 $\beta_0$ 服从均值为 5，标准差为 2 的正态分布,代表：  
  * 当实验条件为 self（编码为 0）时，反应时间的平均值大约为 5 秒。  
  * 截距值可能在 3 ± 7 秒 的范围内波动，反映了在 self 条件下的反应时间预估  
  
* 我们假设 $\beta_1$ 服从均值为 0，标准差为 1 的正态分布，代表：  

  * (斜率)将其均值指定为 1，表示我们预期在 self 和 other 条件下的反应时间差异较小。  

    * 这个影响的量是变化的，范围大概在 -1  ± 1。  
   
* 最后，我们假设 $\sigma$ 服从指数分布，其参数为0.3。  
 
  * 参数0.3 意味着标准差通常集中在较小的正数范围内，使反应时间在预测值$\mu_i$附近波动。  
  * 这一设置允许较小到中等的波动，但大部分数据应集中在 0 到 10 秒的合理反应时间范围内。  

<div style="padding-bottom: 20px;"></div>


可视化指定超参下的先验：  

In [5]:
# 定义先验分布的参数
mu_beta0 <- 5         
sigma_beta0 <- 2     
mu_beta1 <- 0       
sigma_beta1 <- 1   
lambda_sigma <- 0.3      

# 生成 beta_0 的先验分布数据
x_beta0 <- seq(-5, 15, length.out = 1000)
y_beta0 <- dnorm(x_beta0, mean = mu_beta0, sd = sigma_beta0)
df_beta0 <- data.frame(x = x_beta0, y = y_beta0)

# 生成 beta_1 的先验分布数据
x_beta1 <- seq(-5, 5, length.out = 1000)
y_beta1 <- dnorm(x_beta1, mean = mu_beta1, sd = sigma_beta1)
df_beta1 <- data.frame(x = x_beta1, y = y_beta1)

# 生成 sigma 的先验分布数据（指数分布）
x_sigma <- seq(0, 10, length.out = 1000)
y_sigma <- dexp(x_sigma, rate = lambda_sigma)  # R中指数分布参数为rate=λ
df_sigma <- data.frame(x = x_sigma, y = y_sigma)

# 自定义despine函数（作为ggplot图层使用，无需传递参数）
despine <- function() {
  theme(
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    panel.grid.major.y = element_blank(),
    axis.ticks = element_line(color = "black")
  )
}

# 绘制 beta_0 的先验分布
p1 <- ggplot(df_beta0, aes(x = x, y = y)) +
  geom_line(color = "black") +
  ggtitle(expression(N(5, 2^2))) +
  xlab(expression(beta[0])) +
  ylab("pdf") +
  theme_minimal() +
  despine()  # 作为图层直接添加

# 绘制 beta_1 的先验分布
p2 <- ggplot(df_beta1, aes(x = x, y = y)) +
  geom_line(color = "black") +
  ggtitle(expression(N(0, 1^2))) +
  xlab(expression(beta[1])) +
  ylab("pdf") +
  theme_minimal() +
  despine()

# 绘制 sigma 的先验分布
p3 <- ggplot(df_sigma, aes(x = x, y = y)) +
  geom_line(color = "black") +
  ggtitle(expression(Exp(0.3))) +
  xlab(expression(sigma)) +
  ylab("pdf") +
  theme_minimal() +
  despine()


# 组合图形
p1 + p2 + p3 + plot_layout(ncol = 3)

# 显示图形
# print(last_plot())

plot without title

### 先验预测检验(prior predictive check)  

🤔有些同学可能认为这个先验的定义过于随意，甚至有些不靠谱。 那我们是否可以检验先验的合理性，以及适当的调整这个先验呐？  

**我们通过代码来说明，如何进行先验预测检验**  

首先根据公式，先验模型为：  

$$  
\begin{align*}  
\text{priors:} & & \beta_{0}  & \sim N\left(5, 2^2 \right)  \\  
                    & & \beta_1  & \sim N\left(0, 1^2 \right) \\  
                    & & \sigma   & \sim \text{Exp}(0.3)  \\  
\end{align*}  
$$

**先验预测检验的大致思路**  

1. 在先验中随机抽取200组$\beta_0, \beta_1$值  
2. 生成假数据自变量X  
3. 生成200条 $\beta_0 + \beta_1 X$ 生成预测的反应时间数据  
4. 观察生成的反应时间数据是否在合理范围内，评估先验假设的合适性  


1. 在先验中随机抽取200组$\beta_0, \beta_1$值  

In [6]:
# 设置随机种子确保结果可重复
set.seed(84735)

# 根据设定的先验分布，各抽取200个样本
beta0_200 <- rnorm(200, mean = 5, sd = 2)       # 正态分布抽样（均值5，标准差2）
beta1_200 <- rnorm(200, mean = 0, sd = 1)       # 正态分布抽样（均值0，标准差1）
sigma_200 <- rexp(200, rate = 0.3)             # 指数分布抽样（率参数0.3，对应scale=1/0.3）

# 将结果存入数据框
prior_pred_sample <- data.frame(
  beta0 = beta0_200,
  beta1 = beta1_200,
  sigma = sigma_200
)

# 查看抽样结果
head(prior_pred_sample)

,beta0,beta1,sigma
,<dbl>,<dbl>,<dbl>
1,6.334465,0.813496781,5.94110839
2,4.752319,-0.324077783,2.56492873
3,6.561735,0.009246005,0.07751107
4,5.473836,0.363105566,4.89811071
5,6.593022,0.561417222,3.52753446
6,8.166495,-1.316775861,4.95790298


<ol start="2"><li><p>生成假数据自变量X</p></li></ol><ul><li><p>这里我们根据现实情况来定义X的取值范围</p></li></ul><p></p>

In [7]:
# 设置Label，0代表Self，1代表other
x_sim <- c(0, 1)

# 查看自变量值
x_sim

[1] 0 1

3. 根据公式 $\mu = \beta_0 + \beta_1 X$ 生成200条回归线, 观察其中的$\mu$是否处在合理的范围内  

- 我们有200次采样，每次采样都有三个参数 beta_0, beta_1, sigma。  

- 结合每次采样的结果，和自变量X，我们可以生成一条直线  

- 重复这个过程200次，我们就能生成200条直线  



> **我们通过一次采样来理解这个过程**  

![Image Name](https://cdn.kesci.com/upload/smivwvoltb.png?imageView2/0/w/720/h/960)  
- **左侧图表**显示了 200 组随机抽取的参数  beta_0, beta_1, sigma 的值。  
    - 这帮助我们直观地看到参数在各自先验分布下的取值范围。  
- **右侧图表**展示了基于抽取的一个特定参数组合绘制的回归线 $Y = \beta_0 + \beta_1 X$  
    - 红色的点表示预测的反应时间在两个条件下的值，蓝色线是连接这两个预测值的回归线。


## 🎯练习1：先验预测  

根据获取的第一条MCMC链的第一组采样参数，结合自变量X的范围，预测 $\mu$ 的值。  

1. 根据回归公式 $\mu = \beta_0 + \beta_1 X$ 预测$\mu$ 的值。  
2. 绘制回归线条。

In [8]:
# 根据设定的先验分布，各抽取200个样本
beta0_200 <- rnorm(200, mean = 5, sd = 2)       # 正态分布抽样（均值5，标准差2）
beta1_200 <- rnorm(200, mean = 0, sd = 1)       # 正态分布抽样（均值0，标准差1）
sigma_200 <- rexp(200, rate = 0.3)             # 指数分布抽样（率参数0.3）

# 将结果存入数据框
prior_pred_sample <- data.frame(
  beta0 = beta0_200,
  beta1 = beta1_200,
  sigma = sigma_200
)

# 查看抽样结果
# prior_pred_sample

# 获取第一组采样参数
beta_0 <- prior_pred_sample$beta0[1]
beta_1 <- prior_pred_sample$beta1[1]

# 打印第一组采样参数值（保留两位小数）
cat(sprintf("获取的第一组采样参数值，beta_0:%.2f, beta_1:%.2f\n", beta_0, beta_1))

获取的第一组采样参数值，beta_0:2.52, beta_1:-0.03


In [9]:
#===========================
# 根据回归公式 μ = β₀ + β₁X 预测μ的值
# 已知：自变量（标签），self = 1, other = 2
#===========================
x_sim <- c(1, 2)
mu <- beta_0 + beta_1 * x_sim
cat("预测值 μ:", mu, "\n")

预测值 μ: 2.490716 2.459395 


In [10]:
#===========================
# 绘制回归线（完善练习部分）
#===========================
# 准备绘图数据（将x和对应的mu值组合成数据框）
plot_data <- data.frame(
  x_axis = ...,
  y_axis = ...
)

# 绘制回归线
ggplot(plot_data, aes(x = x_axis, y = y_axis)) +
  geom_line(color = "black", linewidth = 1) +  # 绘制回归线
  geom_point(color = "black", size = 3) +      # 添加数据点
  xlab("Label Condition") +                   # x轴标签
  ylab("RT (sec)") +                          # y轴标签
  theme_minimal() +
  # 移除顶部和右侧边框（模拟sns.despine效果）
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major = element_blank(),
    axis.line = element_line(color = "black")
  )

ERROR: Error: '...' used in an incorrect context


In [11]:
#===========================
# 绘制回归线（完善练习部分）
#===========================
# 准备绘图数据（将x和对应的mu值组合成数据框）
plot_data <- data.frame(
  x_axis = x_sim,
  y_axis = mu
)

# 绘制回归线
ggplot(plot_data, aes(x = x_axis, y = y_axis)) +
  geom_line(color = "black", linewidth = 1) +  # 绘制回归线
  geom_point(color = "black", size = 3) +      # 添加数据点
  xlab("Label Condition") +                   # x轴标签
  ylab("RT (sec)") +                          # y轴标签
  theme_minimal() +
  # 移除顶部和右侧边框（模拟sns.despine效果）
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major = element_blank(),
    axis.line = element_line(color = "black")
  )

# 显示图形
# print(last_plot())

plot without title

> 重复上述结果200遍，我们就能得到200次先验预测回归线了  

**可视化先验预测结果**  

- 每一条线代表一次抽样生成的预测，因此我们绘制了200条线。  

- 我们可以观察到 self 和 other 条件下的反应时间如何随着自变量（标签Label）变化。  

- 如果先验设置不合理（如过于宽泛的分布），可能导致预测结果在合理范围之外。  
    - 例如，如果我们将 beta_1 设得过大，可能导致预测的 other 条件下的反应时间显著增加或减少，不符合实验数据的预期。  
    - 因此，通过合理的先验设定，我们可以得到更加符合实验背景的预测结果，这有助于模型对真实数据的拟合。

In [12]:
# 设置实验条件的取值范围，self=0，other=1
x_sim <- c(0, 1)

# 初始化空列表储存预测结果
mu_outcome <- list()

# 循环生成200次先验预测回归线
for (i in 1:nrow(prior_pred_sample)) {
  mu <- prior_pred_sample$beta0[i] + prior_pred_sample$beta1[i] * x_sim
  mu_outcome[[i]] <- mu
}

# 生成200种不同的颜色（使用hcl色空间，确保颜色差异明显）
n_lines <- length(mu_outcome)
colors <- hcl(
  h = seq(0, 360, length.out = n_lines + 1)[-1],  # 色相从0到360度循环
  c = 60,                                         # 饱和度
  l = 60,                                         # 亮度
  alpha = 0.6                                     # 半透明，避免重叠过深
)

# 准备绘图数据
plot_data <- data.frame(
  x = rep(x_sim, n_lines),
  y = unlist(mu_outcome),
  line_id = factor(rep(1:n_lines, each = length(x_sim)))  # 转换为因子用于分组着色
)

# 绘制带不同颜色的先验预测回归线
ggplot(plot_data, aes(x = x, y = y, group = line_id)) +
  geom_line(aes(color = line_id), linewidth = 0.5) +  # 按line_id分配颜色
  scale_color_manual(values = colors) +               # 使用自定义颜色
  ggtitle("prior predictive check") +
  xlab("Label Condition") +
  ylab("RT (sec)") +
  theme_minimal() +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major = element_blank(),
    axis.line = element_line(color = "black"),
    legend.position = "none"  # 隐藏图例（200条线的图例无意义）
  )

plot without title

### 我们的先验设置的合理吗？  

让我们重新聚焦于我们的研究问题：**在知觉匹配任务中，自我相关信息是否会促进认知加工？** 具体而言，我们探讨自我和他人条件下的认知加工差异，尤其是在反应时间上的表现。  

变量定义：  

- 𝑋：标签（Label）  
    - 在自我匹配范式任务中，标签分为 self 和 other 两种，分别编码为 0 和 1。这些条件用于观察在自我相关和非自我相关条件下的反应时间差异。  

- 𝑌：反应时间（RT）  
    - 表示在 self 或 other 条件下参与者的平均反应时间，通常以秒为单位。我们的目标是通过模型预测 self 和 other 条件下反应时间的差异，并观察其随实验条件的变化。  


In [13]:
prior_predictive_plot <- function(beta0_mean = 0.5, beta0_sd = 0.3, 
                                 beta1_mean = -0.1, beta1_sd = 0.04, 
                                 sigma_rate = 0.2, samples = 200, seed = 84735) {
  # 生成先验预测图。
  # 
  # 参数：
  # - beta0_mean: 数值，beta0的均值
  # - beta0_sd: 数值，beta0的标准差
  # - beta1_mean: 数值，beta1的均值
  # - beta1_sd: 数值，beta1的标准差
  # - sigma_rate: 数值，控制sigma的指数分布率参数（lambda = 1/scale）
  # - samples: 整数，生成的样本数量
  # - seed: 整数，随机种子，默认为84735，确保结果可重复
  # 
  # 输出：
  # - 一个先验预测图
  
  # 设置随机种子
  if (!is.null(seed)) {
    set.seed(seed)
  }
  
  # 根据设定的先验分布抽样
  beta0_samples <- rnorm(samples, mean = beta0_mean, sd = beta0_sd)
  beta1_samples <- rnorm(samples, mean = beta1_mean, sd = beta1_sd)
  sigma_samples <- rexp(samples, rate = sigma_rate)
  
  # 创建数据框存储样本
  prior_pred_sample <- data.frame(
    beta0 = beta0_samples,
    beta1 = beta1_samples,
    sigma = sigma_samples
  )
  
  # 定义实验条件（self=0，other=1）
  x_sim <- c(0, 1)
  
  # 生成先验预测结果
  mu_outcome <- lapply(1:samples, function(i) {
    prior_pred_sample$beta0[i] + prior_pred_sample$beta1[i] * x_sim
  })
  
  # 生成多样化颜色
  colors <- hcl(
    h = seq(0, 360, length.out = samples + 1)[-1],
    c = 60,
    l = 60,
    alpha = 0.6
  )
  
  # 准备绘图数据
  plot_data <- data.frame(
    x = rep(x_sim, samples),
    y = unlist(mu_outcome),
    line_id = factor(rep(1:samples, each = length(x_sim)))
  )
  
  # 绘图
  ggplot(plot_data, aes(x = x, y = y, group = line_id)) +
    geom_line(aes(color = line_id), linewidth = 0.5) +
    scale_color_manual(values = colors) +
    ggtitle("Prior Predictive Check") +
    xlab("Label Condition") +
    ylab("RT (sec)") +
    theme_minimal() +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_blank(),
      axis.line = element_line(color = "black"),
      legend.position = "none",
      plot.title = element_text(hjust = 0.5)  # 标题居中
    ) +
    coord_cartesian(expand = FALSE)  # 移除坐标轴扩展空间
  
  # 显示图形
  print(last_plot())
}

# 使用示例
prior_predictive_plot()

plot without title

<h2>🎯练习2：先验预测</h2><p>🤔请大家判断，下图的先验预测合理吗？</p><p></p><img src="https://cdn.kesci.com/upload/smkk0xc4bq.png?imageView2/0/w/720/h/960" alt="Image Name"><p></p>

在实际实验中，我们知道被试的反应时间（RT）不会小于0秒。然而，当前模型的先验预测图中可能会包含一些小于0的反应时间，这显然不符合逻辑.  

通过以下练习,你可以尝试对三个参数的先验分布进行设置，观察它们对反应时间预测的影响。

In [13]:
#===============================================================
#     请完善代码中...的部分，设置3个参数的值，使先验分布更符合实际情况
#===============================================================
# 调用函数并设置符合实际情况的先验参数
prior_predictive_plot(
  beta0_mean = ...,    # beta0的均值
  beta0_sd = ...,      # beta0的标准差
  beta1_mean = ...,    # beta1的均值
  beta1_sd = ...,     # beta1的标准差
  sigma_rate = ...,      # sigma的指数分布率参数
  samples = 200,
  seed = 84735
)

In [14]:
# 调用函数并设置符合实际情况的先验参数
prior_predictive_plot(
  beta0_mean = 0.8,    # beta0的均值：self条件下的平均反应时约0.8秒（符合常见RT范围）
  beta0_sd = 0.2,      # beta0的标准差：控制self条件下的变异（较小的标准差使先验更集中）
  beta1_mean = 0.1,    # beta1的均值：other条件比self条件平均慢0.1秒（符合自我参照效应）
  beta1_sd = 0.05,     # beta1的标准差：组间差异的变异（较小值表示预期差异稳定）
  sigma_rate = 5,      # sigma的指数分布率参数：对应scale=0.2，控制残差变异（较小残差更合理）
  samples = 200,
  seed = 84735
)

plot without title

<h2>模型拟合</h2><h3>模型定义</h3><p>现在，我们可以结合数据与先验，为参数$ (\beta_0, \beta_1, \sigma) $生成后验模型</p><ul><li><p>之后我们可以使用 <code>brm()</code>函数内置的 MCMC 采样功能 来完成对于模型后验分布的采样过程</p></li></ul><p>在这之前，我们回顾之前对先验与似然的定义：</p><p><strong>先验（prior）</strong></p><ul><li><p></p></li></ul><blockquote><p>$ \beta_{0} \sim N\left(5, 2^2 \right) $</p></blockquote><ul><li><p></p></li></ul><blockquote><p>模型的截距项服从均值为 5，标准差为 2 的正态分布。</p></blockquote><ul><li><p></p></li></ul><blockquote><p>$ \beta_1 \sim N\left(0, 1^2 \right) $</p></blockquote><ul><li><p></p></li></ul><blockquote><p>模型的斜率项，服从均值为 0，标准差为 1 的正态分布。</p></blockquote><ul><li><p></p></li></ul><blockquote><p>$ \sigma \sim \text{Exp}(0.3) $</p></blockquote><ul><li><p></p></li></ul><blockquote><p>代表误差项的标准差，服从参数为 0.3 的指数分布。</p></blockquote><p><strong>似然（likelihood）</strong></p><ul><li><p></p></li></ul><blockquote><p>$ \mu_i = \beta_0 + \beta_1X_i $</p></blockquote><ul><li><p></p></li></ul><blockquote><p>$ Y_i {\sim} N\left(\mu_i, \sigma^2\right) $</p></blockquote><p></p>

In [15]:
# 构建贝叶斯线性模型
linear_model <- brm(
  formula = RT_sec ~ Label,  # 公式：RT_sec ~ beta0 + beta1*Label
  data = df,                 # 数据框（包含Label和RT_sec列）
  family = gaussian(),       # 似然函数：正态分布
  
  # 定义先验分布（对应PyMC的先验设置）
  prior = c(
    prior(normal(5, 2), class = Intercept),  # beta0：截距项，对应Normal(mu=5, sigma=2)
    prior(normal(0, 1), class = b),          # beta1：Label的系数，对应Normal(mu=0, sigma=1)
    prior(exponential(3), class = sigma)     # sigma：误差项，对应Exponential(3)
  ),
  
  # MCMC采样参数
  iter = 6000,               # 总迭代次数（draws + tune = 5000 + 1000）
  warmup = 1000,             # 调参迭代次数（对应tune）
  chains = 4,                # 马尔可夫链数量
  cores = 4,                 # 并行计算核心数（加速采样）
  seed = 84735,              # 随机种子，确保结果可重复
  refresh = 0                # 不输出采样过程信息
)

Compiling Stan program...

Start sampling



2. 在采样结束之后，我们得到采样样本（对应R中`brms`/`rstan`的模型结果）  

- 后验样本可通过`posterior_samples()`函数提取，数据类型为数据框（data.frame）或数组（array）。  

  - 样本按“链”和“采样”顺序排列，例如4条链、每条链5000个有效样本时，总样本量为20000行，前5000行为第1条链，接下来5000行为第2条链，以此类推。  

  - 包含3个变量（列），即3个参数：`b_Intercept`（对应$\beta_0$）、`b_Label`（对应$\beta_1$）、`sigma`。  

    - 我们可以使用`posterior$b_Intercept`提取后验中的$\beta_0$参数（`posterior`为`posterior_samples()`的结果）。  

    - 我们可以使用`posterior$b_Intercept[11]`提取$\beta_0$第一条链中第10个有效采样值（注：R索引从1开始，且前`warmup`个样本已自动丢弃，因此第1条链的第10个有效样本对应总索引11）。

**补充 rstan/brms 后验结果结构介绍**  

在 R 的贝叶斯建模中（如使用 `brms` 或 `rstan`），采样结果的存储和管理方式与 PyMC 的 `InferenceData` 类似，但采用了更贴合 R 数据结构的形式，主要通过数据框、列表或数组存储，便于后续分析和可视化。以下是典型的后验结果结构和内容：  


1. `posterior_samples`（后验样本）  
这是建模结果中最核心的部分，包含了每条链和每次采样的后验分布样本，对应 PyMC 中的 `posterior` 组。对于本例的线性模型，后验样本主要包括：  

- `b_Intercept`: 对应 `beta_0`，即截距参数的后验采样值。  
- `b_Label`: 对应 `beta_1`，即斜率参数（Label 对 RT_sec 的影响）的后验采样值。  
- `sigma`: 残差标准差的后验采样值。  

数据结构通常为数据框或数组，每行代表一个样本，每列代表一个参数，样本按“链”的顺序排列（例如 4 条链时，前 5000 行为第 1 条链，接下来 5000 行为第 2 条链，以此类推）。通过这些样本可以绘制后验分布图，计算均值、中位数及可信区间，分析参数的不确定性。  


2. `sample_stats`（采样统计信息）  
`brms` 或 `rstan` 会自动计算采样过程中的统计量，用于评估 MCMC 链的收敛性和采样质量，对应 PyMC 中的 `sample_stats` 组。常见统计信息包括：  

- `lp__`: 对数后验密度（log-posterior），用于评估样本的相对“优劣”。  
- `rhat`: 链间收敛诊断指标，理想值接近 1，若显著偏离 1 可能表明链未收敛。  
- `ess_bulk` 和 `ess_tail`: 分别为整体和尾部的有效样本量，衡量样本的独立性（值越高越好，通常建议大于 1000）。  
- `divergent__`: 若存在发散样本（divergences），可能表明模型对某些参数区域估计不稳定，需要调整先验或增加采样迭代次数。  

这些统计量可通过 `summary()` 或 `rstan::monitor()` 查看，帮助诊断模型是否需要优化（如增加 `warmup` 或 `iter`）。  


3. `observed_data`（观测数据）  
建模时输入的原始数据（如本例中的 `df$RT_sec` 和 `df$Label`）会被保留在模型对象中，对应 PyMC 的 `observed_data` 组。在 `brms` 中可通过 `model$data` 直接提取，便于后续对比观测值与模型预测值，验证模型拟合效果。  


4. `posterior_predictive`（后验预测分布）  
若使用 `posterior_predict()` 函数生成后验预测样本，结果会存储为矩阵或数组，每行对应一个观测值，每列对应一个后验样本。这些预测值（对应 PyMC 中的 `y_est`）反映了模型对观测数据的“复现能力”，通过与 `observed_data` 对比（如绘制预测值与实际值的散点图），可评估模型的预测准确性和泛化能力。  


5. `prior`（先验分布样本）  
若通过 `prior_samples()` 函数生成先验样本，结果会包含各参数的先验分布采样值，对应 PyMC 的 `prior` 组。分析先验样本可帮助我们判断先验设置是否合理（如是否过度限制参数范围），确保先验不与领域知识冲突。  


这些结构共同构成了 R 中贝叶斯模型的结果体系，通过 `brms` 或 `rstan` 提供的函数（如 `summary()`、`plot()`、`posterior_samples()`）可方便地提取和分析，实现与 PyMC + ArviZ 类似的功能。

In [16]:
# 查看后验分布（对应trace.posterior）
posterior <- posterior_samples(linear_model)
print(head(posterior))

Warning message:
“Method 'posterior_samples' is deprecated. Please see ?as_draws for recommended alternatives.”


  b_Intercept    b_Label     sigma Intercept    lprior       lp__
1   0.6972429 0.19484524 0.2521145 0.8141501 -4.397905 -0.7303698
2   0.7249656 0.10831469 0.2091866 0.7899544 -4.281398 -0.5344268
3   0.7249656 0.10831469 0.2091866 0.7899544 -4.281398 -0.5344268
4   0.7048390 0.13693277 0.2519270 0.7869986 -4.416241 -0.3549063
5   0.7456322 0.08938783 0.2197848 0.7992649 -4.301534 -0.1991421
6   0.7688410 0.08997147 0.2069120 0.8228238 -4.238295 -1.7638298


In [17]:
# 提取beta_0的后验样本（对应trace.posterior['beta_0']）
beta0_posterior <- posterior$b_Intercept
print("beta_0的后验样本（前10个）：")
print(head(beta0_posterior, 10))

[1] "beta_0的后验样本（前10个）："
 [1] 0.6972429 0.7249656 0.7249656 0.7048390 0.7456322 0.7688410 0.7172070
 [8] 0.7235347 0.7547343 0.7538959


In [18]:
# 提取第1条链的第11个样本（R索引从1开始，对应Python的[0,10]）
# 注：brms默认将所有链的样本合并，按顺序排列
chain1_sample11 <- beta0_posterior[11]  # 第1条链的第11个样本（前1000个是warmup，已自动丢弃）
cat(sprintf("第1条链的第11个beta_0样本值：%.4f\n", chain1_sample11))

第1条链的第11个beta_0样本值：0.6920


<h3>MCMC 诊断</h3><p>使用 ` mcmc_trace() ` 可视化参数的后验分布</p>

In [19]:
# 1. 提取后验样本并添加链信息
posterior <- posterior_samples(linear_model) %>%
  select(
    beta0 = b_Intercept,  # 截距项
    beta1 = b_Label,      # Label系数
    sigma = sigma         # 残差标准差
  )

n_chains <- 4                  # 4条链
samples_per_chain <- nrow(posterior) / n_chains  # 每条链样本数

# 标记链编号和迭代序号
posterior_with_chain <- posterior %>%
  mutate(
    chain = rep(1:n_chains, each = samples_per_chain),
    iteration = rep(1:samples_per_chain, times = n_chains)
  )

# 2. 绘制APA格式迹线图（采样轨迹）
trace_data <- posterior_with_chain %>%
  pivot_longer(cols = c(beta0, beta1, sigma), names_to = "parameter", values_to = "value")

trace_plot <- ggplot(trace_data, aes(x = iteration, y = value, color = factor(chain))) +
  geom_line(size = 0.1) +
  facet_wrap(~parameter, ncol = 1, scales = "free_y") +  # 纵向排列参数
  scale_color_brewer(palette = "Set1", name = "Chain") +  # APA推荐配色
  labs(
    x = "Iteration", 
    y = "Parameter Value", 
    title = "MCMC Sampling Traces"
  ) +
  papaja::theme_apa() +  # 应用APA格式主题
  theme(
    plot.title = element_text(hjust = 0.5, size = 12),  # 标题居中
    legend.position = "bottom",                        # 图例在底部
    panel.border = element_rect(color = "black", fill = NA),  # 边框可见
    strip.text.x = element_text(size = 10)             # 分面标签大小
  )

# 3. 绘制APA格式后验分布图（密度曲线）
density_data <- posterior_with_chain %>%
  pivot_longer(cols = c(beta0, beta1, sigma), names_to = "parameter", values_to = "value")

density_plot <- ggplot(density_data, aes(x = value, color = factor(chain), fill = factor(chain))) +
  geom_density(alpha = 0.2, linewidth = 0.8) +  # 线条加粗，符合APA规范
  facet_wrap(~parameter, ncol = 1, scales = "free_x") +
  scale_color_brewer(palette = "Set1", name = "Chain") +
  scale_fill_brewer(palette = "Set1", name = "Chain") +
  labs(
    x = "Parameter Value", 
    y = "Density", 
    title = "Posterior Distributions"
  ) +
  papaja::theme_apa() +  # 应用APA格式主题
  theme(
    plot.title = element_text(hjust = 0.5, size = 12),
    legend.position = "bottom",
    panel.border = element_rect(color = "black", fill = NA),
    strip.text.x = element_text(size = 10)
  )

# 4. 组合图形（2行1列，APA格式布局）
par(mfrow = c(2, 1), mar = c(4, 4, 3, 1))  # 调整边距，符合APA留白规范
print(trace_plot)
print(density_plot)
par(mfrow = c(1, 1))  # 重置布局

Warning message:
“Method 'posterior_samples' is deprecated. Please see ?as_draws for recommended alternatives.”
Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”


plot without title

plot without title

对于模型的诊断信息 ess_bulk 和 r_hat (当然你可以结合可视化进行诊断)。  
- 其中，各参数的 r_hat 均接近于 1；  
- 各参数的ess_bulk 均大于 400，并且有效样本量占比 (6000/20000)=0.3，大于0.1(即10%)。  

In [20]:
# 1. 获取模型完整摘要（兼容所有brms版本）
full_summary <- summary(linear_model)

# 2. 提取固定效应参数（beta0和beta1）的诊断信息
# 动态匹配有效样本量列名（可能是Bulk_ESS或ESS）
fixed_cols <- colnames(full_summary$fixed)
ess_col <- if ("Bulk_ESS" %in% fixed_cols) "Bulk_ESS" else "ESS"

fixed_diag <- as.data.frame(full_summary$fixed) %>%
  select(
    r_hat = Rhat,          # 收敛指标r_hat（列名固定）
    ess_bulk = all_of(ess_col)  # 动态匹配有效样本量列名
  ) %>%
  rownames_to_column("parameter") %>%
  mutate(
    parameter = recode(
      parameter,
      "(Intercept)" = "beta0",  # 截距项重命名
      "Label" = "beta1"         # Label系数重命名
    )
  )

# 3. 提取sigma参数的诊断信息
sigma_cols <- colnames(full_summary$spec_pars)
sigma_ess_col <- if ("Bulk_ESS" %in% sigma_cols) "Bulk_ESS" else "ESS"

sigma_diag <- as.data.frame(full_summary$spec_pars) %>%
  select(
    r_hat = Rhat,
    ess_bulk = all_of(sigma_ess_col)
  ) %>%
  mutate(parameter = "sigma")  # 指定sigma参数名

# 4. 合并诊断信息并计算有效样本量占比
diagnostics <- bind_rows(fixed_diag, sigma_diag) %>%
  mutate(
    # 提取尾部有效样本量（动态匹配列名）
    ess_tail = if ("Tail_ESS" %in% fixed_cols) {
      c(full_summary$fixed[, "Tail_ESS"], full_summary$spec_pars[, "Tail_ESS"])
    } else {
      ess_bulk  # 若不存在则用批量有效样本量近似
    },
    ess_bulk_ratio = ess_bulk / 20000  # 总采样量=4链×5000=20000
  ) %>%
  select(parameter, r_hat, ess_bulk, ess_tail, ess_bulk_ratio)

# 显示诊断结果
print(diagnostics)

      parameter    r_hat ess_bulk ess_tail ess_bulk_ratio
...1  Intercept 1.000065 20509.81 14172.48      1.0254906
...2      beta1 1.000115 19820.13 14815.40      0.9910065
sigma     sigma 1.000871 16626.64 14448.11      0.8313319


## 后验预测  

MCMC 诊断仅能显示采样过程的收敛性和采样质量，而不能直接说明模型在观测数据上的拟合效果或预测能力。为评估模型的预测能力，我们可以进行**后验预测检查**，通过模拟新的数据样本来比较模型生成的数据与实际观测数据的分布是否一致，从而验证模型的合理性和适用性。  

- 为了更全面地反映模型的不确定性，我们可以基于 20000 对参数值生成 20000 条回归线。  

- 这些回归线将展示在 "self" 和 "other" 条件下的预测差异，以及模型在不同参数样本下的预测范围。这种基于后验参数的预测被称为后验预测 (Posterior Prediction)。  

- 在进行后验预测时，我们利用模型后验采样得到的参数进行预测，并结合真实数据生成预测值的分布。  
- 这个过程不仅可以帮助我们检查模型对数据的适配度，还能通过可视化展现预测的不确定性。

### 🎯练习  

根据 **先验预测检验可视化预测结果**的思路，对于后验预测结果进行可视化。  

1. 使用真实数据中的自变量Label  
2. 根据 20000对参数（beta_0, beta_1），与自变量(Label)进行组合，生成了20000条回归线  
3. 绘制后验预测结果  


In [21]:
# 定义x轴代表的Label（0=Self，1=Other）
x_sim <- c(0, 1)

# 提取后验样本并转换为长格式
posterior <- posterior_samples(linear_model) %>%
  select(b_Intercept, b_Label) %>%  # 选择beta0和beta1
  rename(beta0 = b_Intercept, beta1 = b_Label)

# 选取前2个样本用于预测（对应Python代码的[:2]）
beta_0 <- head(posterior$beta0, 2)
beta_1 <- head(posterior$beta1, 2)

# 生成回归线（每个样本对应一条线）
y_sim_re <- lapply(1:2, function(i) {
  beta_0[i] + beta_1[i] * x_sim
})

# 转换为绘图数据框
pred_data <- data.frame(
  x = rep(x_sim, 2),
  y = unlist(y_sim_re),
  sample = factor(rep(1:2, each = length(x_sim)))
)

# 提取观测数据（真实数据）
observed_data <- data.frame(
  x = df$Label,  # 原始Label数据
  y = df$RT_sec  # 原始反应时间
)

# 绘制后验预测检查图
ggplot() +
  # 绘制真实数据散点图
  geom_point(data = observed_data, aes(x = x, y = y), 
             color = "red", alpha = 0.6, size = 2, label = "observed data") +
  # 绘制回归线
  geom_line(data = pred_data, aes(x = x, y = y, group = sample), 
            color = "grey50", linewidth = 1) +
  # 绘制回归线端点
  geom_point(data = pred_data, aes(x = x, y = y), 
             color = "black", size = 3) +
  # 设置坐标轴范围和刻度
  xlim(-0.5, 1.5) +
  scale_x_continuous(breaks = c(0, 1)) +
  # 添加标题和标签
  ggtitle("posterior predictive check") +
  xlab("Label") +
  ylab("RT (sec)") +
  # 添加图例
  scale_color_manual(values = c("red", "grey50", "black"),
                     labels = c("observed data", "Predicted Mean", "")) +
  guides(color = guide_legend(override.aes = list(
    shape = c(16, NA, 16),
    linetype = c(0, 1, 0)
  ))) +
  # 美化主题
  theme_minimal() +
  theme(
    panel.grid = element_blank(),
    axis.line = element_line(color = "black"),
    legend.position = "bottom"
  )

# 显示图形
# print(last_plot())

Warning message:
“Method 'posterior_samples' is deprecated. Please see ?as_draws for recommended alternatives.”
Warning message in geom_point(data = observed_data, aes(x = x, y = y), color = "red", :
“Ignoring unknown parameters: `label`”
Scale for x is already present.
Adding another scale for x, which will replace the existing scale.


plot without title

<p><strong>使用ggplot2绘制后验预测的线性模型</strong></p><p><strong>代码详解</strong></p><ul><li><p>与上一段代码最大的不同之处在于，此时需要基于后验样本生成<code>y_model</code>预测值，并计算其均值和可信区间</p></li><li><p>在绘图逻辑中:</p><ul><li><p><code>y</code> 为真实数据中按<code>Label</code>分组的均值（<code>df['Mean RT']</code>）</p></li><li><p><code>x</code> 为真实数据中的自变量<code>df$Label</code>（0代表Self，1代表Other）</p></li><li><p><code>y_model</code> 为结合后验采样生成的预测值集合，通过以下元素在图中展示：</p><ul><li><p>灰色阴影区域：表示预测值的95%可信区间（不确定性范围）</p></li><li><p>黑色实线：表示后验预测的均值回归线</p></li><li><p>黑色散点：表示真实数据的分组均值</p></li></ul></li></ul></li></ul><blockquote><p>😎<em>通过向量化计算优化，运行效率较高</em></p></blockquote><p></p>

In [22]:
# 计算观测数据的分组均值
df <- df %>%
  group_by(Label) %>%
  mutate(`Mean RT` = mean(RT_sec)) %>%
  ungroup()

# 提取后验样本中的beta0和beta1
posterior <- posterior_samples(linear_model) %>%
  select(b_Intercept, b_Label) %>%
  rename(beta0 = b_Intercept, beta1 = b_Label)

# 生成y_model预测值
x_values <- c(0, 1)
y_model <- lapply(1:nrow(posterior), function(i) {
  posterior$beta0[i] + posterior$beta1[i] * x_values
})

# 转换为数据框并计算均值和95%可信区间
y_model_df <- do.call(rbind, y_model) %>%
  as.data.frame() %>%
  setNames(paste0("x=", x_values)) %>%
  pivot_longer(everything(), names_to = "x", values_to = "y") %>%
  mutate(x = as.numeric(sub("x=", "", x))) %>%
  group_by(x) %>%
  summarize(
    mean = mean(y),
    lower = quantile(y, 0.025),
    upper = quantile(y, 0.975)
  )

# 提取观测均值数据
observed_mean <- df %>%
  select(Label, `Mean RT`) %>%
  distinct() %>%
  rename(x = Label, y = `Mean RT`)

# 绘制后验预测线性模型（修正图例设置）
ggplot() +
  # 不确定性区间
  geom_ribbon(data = y_model_df, 
              aes(x = x, ymin = lower, ymax = upper, fill = "Uncertainty in mean"), 
              alpha = 0.5) +
  # 后验均值线
  geom_line(data = y_model_df, 
            aes(x = x, y = mean, color = "Mean"), 
            linewidth = 2) +
  # 观测均值点
  geom_point(data = observed_mean, 
             aes(x = x, y = y, color = "observed mean"), 
             size = 3) +
  # 坐标轴设置
  xlim(-0.5, 1.5) +
  scale_x_continuous(breaks = c(0, 1)) +
  xlab("Label") +
  ylab("RT (sec)") +
  # 图例样式设置（修正order参数错误）
  scale_fill_manual(values = "grey70", name = NULL) +
  scale_color_manual(values = c("black", "black"), name = NULL) +
  guides(
    fill = guide_legend(order = 2),
    color = guide_legend(order = 1,  # 单个order值，解决尺寸错误
                        override.aes = list(
                          shape = c(16, NA),  # 观测点为圆点，线为无形状
                          linetype = c(0, 1)  # 观测点无线条，线为实线
                        ))
  ) +
  # 主题设置
  theme_minimal() +
  theme(
    panel.grid = element_blank(),
    axis.line = element_line(color = "black"),
    legend.position = "bottom",
    text = element_text(size = 16)
  )

# print(last_plot())

Warning message:
“Method 'posterior_samples' is deprecated. Please see ?as_draws for recommended alternatives.”
Scale for x is already present.
Adding another scale for x, which will replace the existing scale.


plot without title

### 通过MCMC采样值理解后验预测分布  


* 通过MCMC采样，三个参数各获得了20000个采样值$\left(\beta_0^{(i)},\beta_1^{(i)},\sigma^{(i)}\right)$  

* 根据 20000 组参数值 $\beta_0$ 和 $\beta_1$，可以得到 20000 个均值 $\mu$ 的可能值。然后再根据 $\mu$ 生成预测值 $Y_{\text{new}}$。  
* 20000 个均值 $\mu$ 构成了预测的均值分布：  

$$  
\left[  
\begin{array}{ll}  
\beta_0^{(1)} & \beta_1^{(1)} \\  
\beta_0^{(2)} & \beta_1^{(2)} \\  
\vdots & \vdots \\  
\beta_0^{(20000)} & \beta_1^{(20000)} \\  
\end{array}  
\right]  
\;\; \longrightarrow \;\;  
\left[  
\begin{array}{l}  
\mu^{(1)} \\  
\mu^{(2)} \\  
\vdots \\  
\mu^{(20000)} \\  
\end{array}  
\right]  
$$  

- 为了模拟这个过程，我们首先从后验分布中提取采样结果，并生成每个采样值对应的预测均值 $\mu$。每个均值 $\mu^{(i)}$ 可以通过以下公式计算：  

$$  
\mu^{(i)} = \beta_0^{(i)} + \beta_1^{(i)} X  
$$  

- 然后，在每个均值 $\mu^{(i)}$ 的基础上，加入噪声项 $\epsilon$ 来生成 $Y_{\text{new}}^{(i)}$：  

$$  
Y_{\text{new}}^{(i)} = \mu^{(i)} + \epsilon^{(i)}, \quad \epsilon^{(i)} \sim \mathcal{N}(0, \sigma^{(i)})  
$$  

- 这里，$\epsilon^{(i)}$ 是服从均值为 0，方差为 $\sigma^{(i)}$ 的正态分布。  

> 可以注意到，生成的预测值受到两种变异的影响：  
> * 一是参数估计的不确定性（即 $\beta_0$ 和 $\beta_1$ 的后验分布带来的变异），导致不同样本的均值 $\mu$ 具有差异；  
> * 二是随机误差项 $\epsilon$ 的影响，使得在相同均值 $\mu$ 下生成的预测值 $Y_{\text{new}}$ 仍然存在随机波动。这两种变异共同决定了最终预测值的后验预测分布。  

<div style="padding-bottom: 30px;"></div>


### 提取后验样本并生成预测  

**我们也可以用代码来进行模拟，首先我们先进行单次完整的抽取过程**

In [23]:
# 提取后验样本并转换为数据框（包含所有链和采样结果）
# brms的posterior_samples()已自动合并所有链的样本，共4链×5000采样=20000个样本
df_pos_sample <- posterior_samples(linear_model) %>%
  # 选择需要的参数并按原代码命名
  select(
    beta_0 = b_Intercept,  # 截距项对应beta_0
    beta_1 = b_Label,      # 斜率项对应beta_1
    sigma = sigma          # 残差标准差
  )

# 查看参数数据框
df_pos_sample

Warning message:
“Method 'posterior_samples' is deprecated. Please see ?as_draws for recommended alternatives.”


beta_0,beta_1,sigma
<dbl>,<dbl>,<dbl>
0.6972429,0.19484524,0.2521145
0.7249656,0.10831469,0.2091866
0.7249656,0.10831469,0.2091866
0.7048390,0.13693277,0.2519270
0.7456322,0.08938783,0.2197848
0.7688410,0.08997147,0.2069120
0.7172070,0.15377147,0.2212002
0.7235347,0.08236067,0.2383283
0.7547343,0.05359633,0.2490419


In [24]:
# 抽取第一组参数组合（R索引从1开始，对应Python的row_i=0）
row_i <- 1  
X_i <- 1   

# 计算正态分布的均值mu_i
mu_i <- df_pos_sample$beta_0[row_i] + df_pos_sample$beta_1[row_i] * X_i           
sigma_i <- df_pos_sample$sigma[row_i]

# 从正态分布中随机抽取一个值，作为预测值
prediction_i <- rnorm(n = 1, mean = mu_i, sd = sigma_i)

# 打印结果（可多次运行，观察相同参数下的预测值变化）
cat(sprintf("mu_i: %.2f, 预测值：%.2f\n", mu_i, prediction_i))

mu_i: 0.89, 预测值：0.85


**使用代码模拟多次后验预测**  

* 通过上述四行代码，我们已经进行了一次完整的后验预测  

* 我们可以写一个循环，重复这个过程20000次  

* 最后的结果中，每一行代表一个参数对；mu 为预测的均值，y_new 为实际生成的预测值。

In [25]:
# 生成两个空列，用于储存均值mu和预测值y_new
df_pos_sample$mu <- NA
df_pos_sample$y_new <- NA

# 设置X_i的值和随机种子（保持与原代码一致）
X_i <- 1
set.seed(84735)

# 循环计算均值并生成预测值（共20000次，与后验样本数量一致）
for (row_i in 1:nrow(df_pos_sample)) {
  # 计算均值mu_i
  mu_i <- df_pos_sample$beta_0[row_i] + df_pos_sample$beta_1[row_i] * X_i
  df_pos_sample$mu[row_i] <- mu_i
  
  # 从正态分布中抽取预测值y_new
  df_pos_sample$y_new[row_i] <- rnorm(
    n = 1,
    mean = mu_i,
    sd = df_pos_sample$sigma[row_i]
  )
}

# 查看结果（可选）
head(df_pos_sample)

,beta_0,beta_1,sigma,mu,y_new
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.6972429,0.19484524,0.2521145,0.8920882,1.0603072
2,0.7249656,0.10831469,0.2091866,0.8332803,0.8073746
3,0.7249656,0.10831469,0.2091866,0.8332803,0.9966274
4,0.7048390,0.13693277,0.2519270,0.8417717,0.9014578
5,0.7456322,0.08938783,0.2197848,0.8350200,1.0100811
6,0.7688410,0.08997147,0.2069120,0.8588124,1.1864053


**绘制后验预测分布**  

根据刚刚生成的数据，我们可以分别绘制出 $\mu$ 与 $Y_{new}$ 的后验预测分布图

In [26]:
# 复制数据框
df2 <- df %>%
  filter(Label == 1)      # 筛选Label=1的行

# 查看x=1时y的取值
cat("x=1时y的取值有:", "\n")
print(df2$RT_sec)  # 输出Label=1对应的RT_sec值

x=1时y的取值有: 
 [1] 0.753 0.818 0.917 0.717 0.988 0.950 0.657 0.829 1.143 0.756 0.665 0.846
[13] 0.839 0.914 0.712 1.330 0.786 0.626 0.912 0.725 0.956 0.485 1.417 0.604
[25] 0.789 1.327 1.357 0.635 0.871 1.287 0.739 1.331 0.907 1.015 1.125 0.868
[37] 0.582 1.233 1.030 0.791 1.028 0.918 0.793 0.909 0.646 0.467 0.843 0.610
[49] 0.972 0.851 1.208 0.473 0.407 1.416 1.164 0.605 1.071 0.425 0.634 0.393
[61] 1.020 0.414 0.698


In [27]:
# 计算X轴全局范围（覆盖mu和y_new）
x_min <- min(df_pos_sample$mu, df_pos_sample$y_new)
x_max <- max(df_pos_sample$mu, df_pos_sample$y_new)

# 计算Y轴全局范围（覆盖两个分布的密度最大值）
# 先分别计算两个分布的密度值
density_mu <- density(df_pos_sample$mu)
density_ynew <- density(df_pos_sample$y_new)
y_max <- max(density_mu$y, density_ynew$y)  # 取密度最大值
y_min <- 0  # 密度从0开始

# 设置图形布局（1行2列）
par(mfrow = c(1, 2))

# 第一个图：mu的分布（统一X和Y轴范围）
p1 <- ggplot(df_pos_sample, aes(x = mu)) +
  geom_density(color = "black", fill = "grey80", alpha = 0.5) +
  xlim(x_min, x_max) +  # 统一X轴
  ylim(y_min, y_max) +  # 统一Y轴
  ggtitle("mu distribution") +
  xlab("Value") +
  ylab("Density") +
  theme_minimal() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black")
  )

# 第二个图：y_new的分布（完全一致的轴范围）
p2 <- ggplot(df_pos_sample, aes(x = y_new)) +
  geom_density(color = "black", fill = "grey80", alpha = 0.5) +
  xlim(x_min, x_max) +  # 与第一个图X轴一致
  ylim(y_min, y_max) +  # 与第一个图Y轴一致
  ggtitle("y_new distribution") +
  xlab("Value") +
  ylab("Density") +
  theme_minimal() +
  theme(
    plot.title = element_text(hjust = 0.5),
    panel.grid = element_blank(),
    axis.line = element_line(color = "black")
  )

# 显示图形
print(p1)
print(p2)

# 重置图形布局
par(mfrow = c(1, 1))

plot without title

plot without title

从上图可以看到， $Y_{new}$ 分布的不确定性远大于 $\mu$ 分布的不确定性：  

- $\mu$ 分布窄且集中，反映了模型的稳定预测中心；  
- 而 $Y_{new}$ 分布较宽，反映了模型的不确定性。  

> 正如之前提到那样，生成的预测值受到两种变异的影响：  
> * 一是参数估计的不确定性（即 $\beta_0$ 和 $\beta_1$ 的后验分布带来的变异），导致不同样本的均值 $\mu$ 具有差异；  
> * 二是从分布到数据中，另一个参数 $\sigma$ 的影响，进一步放大了预测值 $Y_{\text{new}}$ 的不确定性。这两种变异共同决定了最终预测值的后验预测分布。

<h3>总体后验预测分布</h3><ul><li><p>除了生成特定自变量下，因变量的分布，也可以生成总体因变量的后验预测分布</p></li><li><p>通过 <code>posterior_predic</code>方法可以快速从模型生成后验预测数据。</p></li></ul><p></p>

In [28]:
# 基于模型和后验样本生成后验预测分布
ppc_data <- posterior_predict(linear_model)
# 查看后验预测结果
ppc_data

0.8805146,0.5461327,1.0734812,1.1180581,1.0544820,1.1436725,1.3296152,0.8377308,1.1372116,1.5549294,⋯,0.6550912,0.1358554,0.2363826,0.5534482,0.49387311,0.9364979,0.3121071,0.5023882,0.7732767,0.6522038
0.8491705,0.4609692,1.0697317,0.9741181,0.6590621,0.9564992,0.7785661,0.7777750,0.7369499,0.4912747,⋯,0.2576169,0.6296230,0.5259717,0.8240971,0.56916557,1.0031616,0.5863618,1.0058955,0.7054057,0.8460817
0.9303563,0.6071321,1.0619388,0.6525205,0.4943956,0.9004989,1.1434786,0.6188290,0.6659093,0.8614999,⋯,1.2200525,0.8686903,0.4135834,0.6594837,0.72630167,0.7115717,0.9367772,0.8057101,0.4445820,0.3588234
0.7398532,1.1925242,0.9668437,0.8961170,0.8121803,0.3493240,1.0862478,1.2433553,0.8471879,0.5889876,⋯,1.0661684,0.8589036,1.0354501,0.5104175,0.34756218,0.6470245,0.7250589,0.9558478,1.1506028,1.0765313
0.8463258,0.8097733,0.7255550,0.9554507,0.4846799,0.4091627,0.8621464,0.6495574,0.9875643,0.6707358,⋯,0.5036867,1.1774088,0.4781792,0.7114487,0.79123133,0.7757159,0.7737694,0.4846296,0.6908176,0.5601552
1.0274248,0.9045376,0.7619572,0.8398848,1.2034348,0.7317004,1.0047085,0.6009903,0.9441062,0.9680857,⋯,0.6963298,0.8244431,0.7663458,0.7673247,0.93697595,0.5763842,0.7594903,0.6731668,0.5266829,0.4822042
0.9648705,0.5095762,1.1322569,0.3448469,0.8358403,0.7553396,0.8865015,0.7972612,0.4069173,0.9963015,⋯,0.4717066,0.4545913,0.7421432,0.6875413,0.45973645,0.7762252,0.9558091,0.6670006,0.7807792,0.8177253
0.4863375,0.9652378,0.4515244,0.8968525,0.9258446,0.8354505,0.4706630,0.6330778,0.8838873,0.7888660,⋯,0.5103332,0.6070879,0.4636683,0.5307378,0.35781982,0.6087352,0.6030195,0.5286985,1.1758749,0.4583638
0.5746304,0.8817290,0.7984690,0.9942958,0.6106121,0.8530169,0.5849666,0.8966153,0.7822224,0.5399022,⋯,0.7925758,0.9314801,0.9828676,0.6665150,0.98182535,0.6850709,0.6531674,0.4090336,0.2192221,0.5598452
0.8183124,0.8483435,1.1323427,0.8434898,0.4619965,0.7038110,0.8969503,1.1356299,0.5207586,0.8399301,⋯,0.8082129,0.8101556,0.6915985,0.8236921,1.36417180,0.8385718,1.1524545,0.7026104,1.0808821,1.0786068
1.0915488,1.0996779,1.1291896,0.7382508,0.8046575,0.9452956,0.6664163,0.8098866,1.1441313,0.9956693,⋯,0.2606664,1.0027979,0.8909334,0.7648952,1.02120845,1.0321806,0.5673541,0.9650042,0.5152721,1.1698699


<p>输出说明：返回一个矩阵，行对应原始数据的每个观测值，列对应后验样本（共 20000 个），每个元素表示该观测在某一后验样本下的预测值。<br>接着，我们可以使用 brms 提供的后验预测检查函数 pp_check() 来绘制结果：<br>- 黑色线条代表观测值（RT_sec）的总体分布情况。<br>- 蓝色线条代表 300 个后验预测样本各自的分布情况。</p><p>- 橙色线条代表后验预测的均值的分布情况。</p><p></p>

In [29]:
#-------------------------------------------------------
# 1. pp_check: 蓝色 posterior predictive + 黑色 observed
#-------------------------------------------------------
p <- pp_check(
  linear_model,
  type = "dens_overlay",
  ndraws = 300
)

#-------------------------------------------------------
# 2. posterior predictive draws
#    结构： (draw × n_obs)
#-------------------------------------------------------
pp_samples <- posterior_predict(linear_model)

#-------------------------------------------------------
# 3. 计算“平均 posterior predictive density”
#   对每个 draw 单独跑 density，然后对 y 值取 average
#-------------------------------------------------------
dens_list <- apply(pp_samples, 1, density)

# x 轴来自第一条 density（所有 density 的 x 都一致）
x_vals <- dens_list[[1]]$x

# y 轴为所有 density 的平均
y_vals <- Reduce("+", lapply(dens_list, function(d) d$y)) / length(dens_list)

df_avg <- data.frame(x = x_vals, y = y_vals)

#-------------------------------------------------------
# 4. 把橙色虚线的“平均 posterior predictive density”叠加到 pp_check 图上
#-------------------------------------------------------
p +
  geom_line(
    data = df_avg,
    aes(x = x, y = y),
    color = "orange",
    linetype = "dashed",
    linewidth = 1
  ) +
  labs(
    title = "Posterior Predictive Check with Mean Predictive Density",
    x = "RT",
    y = "Density"
  ) +
  theme_bw(base_size = 14)


plot without title

<h3>对新数据的预测</h3><ul><li><p>采样得到的后验参数基于编号为"201"的被试数据，到目前为止，我们都在使用后验参数对这一批数据做出后验预测</p></li><li><p>那么基于编号为"201"的被试数据得出的后验参数估计对其他数据的预测效果如何？</p></li><li><p>我们可以选用一批新的数据，查看当前参数是否能预测新数据(例如 "205")中的变量关系</p></li></ul><p></p>

In [33]:
# 1. 预处理：统一 Label 编码为 0 / 1
df <- df_raw %>%
  mutate(
    Subject = as.character(Subject),
    Label = case_when(
      Label == 1 ~ 0,
      Label == 2 ~ 1,
      Label == 3 ~ 1,
      TRUE ~ Label   # 其他值保持（若数据结构不符，可再处理）
    )
  )

# 筛选特定被试和条件的数据
df_201 <- df %>% filter(Subject == "201", Matching == "Matching") %>% select(Label, RT_sec)
df_205 <- df %>% filter(Subject == "205", Matching == "Matching") %>% select(Label, RT_sec)

# 稳健后验整理与 APA 绘图函数（含灰色边界线）
plot_posterior_apa <- function(model, df_sub, title = "Posterior Predictive") {
  # 1) 计算观测均值及误差
  df_mean <- df_sub %>%
    group_by(Label) %>%
    summarise(
      Mean_RT = mean(RT_sec),
      SD_RT = sd(RT_sec),
      N = n(),
      SE_RT = SD_RT / sqrt(N),
      .groups = "drop"
    ) %>%
    arrange(Label)
  
  # 2) 生成后验预测
  linpred <- posterior_linpred(model, newdata = df_mean, transform = TRUE)
  
  # 3) 处理后验预测矩阵
  linpred_mat <- as.matrix(linpred)
  if (is.null(colnames(linpred_mat)) || any(colnames(linpred_mat) == "")) {
    colnames(linpred_mat) <- paste0("V", seq_len(ncol(linpred_mat)))
  }
  
  # 4) 转换为长格式数据
  pred_df <- as.data.frame(linpred_mat)
  pred_df$draw <- seq_len(nrow(pred_df))
  
  pred_long <- pred_df %>%
    pivot_longer(
      cols = -draw,
      names_to = "LabelIndex",
      values_to = "y_model"
    ) %>%
    mutate(
      idx = as.integer(gsub("\\D", "", LabelIndex)),
      Label = df_mean$Label[idx]
    )
  
  # 5) 汇总后验统计量
  pred_summary <- pred_long %>%
    group_by(Label) %>%
    summarise(
      y_mean = mean(y_model),
      y_lower = quantile(y_model, 0.025),
      y_upper = quantile(y_model, 0.975),
      .groups = "drop"
    ) %>%
    arrange(Label)
  
  # 6) 绘图（含灰色边界线）
  p <- ggplot() +
    # 后验95%区间（带灰色边界）
    geom_ribbon(
      data = pred_summary,
      aes(x = Label, ymin = y_lower, ymax = y_upper),
      alpha = 0.25,          # 填充透明度
      fill = "gray70",       # 填充色
      color = "gray50",      # 边界线颜色（灰色）
      linewidth = 0.6        # 边界线粗细
    ) +
    # 后验均值线
    geom_line(
      data = pred_summary,
      aes(x = Label, y = y_mean),
      linewidth = 1
    ) +
    # 观测均值点
    geom_point(
      data = df_mean,
      aes(x = Label, y = Mean_RT),
      size = 3
    ) +
    # 观测均值误差条
    geom_errorbar(
      data = df_mean,
      aes(x = Label, ymin = Mean_RT - SE_RT, ymax = Mean_RT + SE_RT),
      width = 0.05,
      linewidth = 0.8
    ) +
    # APA风格主题
    papaja::theme_apa() +
    labs(
      x = "Label (0 = Self, 1 = Other)",
      y = "Reaction Time (s)",
      title = title
    ) +
    theme(
      plot.title = element_text(face = "bold", size = 14),
      axis.title = element_text(size = 12),
      axis.text = element_text(size = 10)
    ) +
    scale_x_continuous(breaks = c(0, 1), limits = c(-0.1, 1.1)) +
    scale_y_continuous(limits = c(0.65, 0.95))
  
  return(p)
}

# 生成两个被试的图
p1 <- plot_posterior_apa(linear_model, df_201, "Subject 201")
p2 <- plot_posterior_apa(linear_model, df_205, "Subject 205")

# 并排显示
library(patchwork)  # 确保已安装patchwork包用于组合图形
(p1 + p2) & theme(plot.margin = unit(rep(8, 4), "pt"))

Warning message:
“Removed 1 row containing missing values or values outside the scale range
(`geom_ribbon()`).”
Warning message:
“Removed 1 row containing missing values or values outside the scale range
(`geom_ribbon()`).”


plot without title

<h2>后验推断</h2><p>我们共得到20000对$ \beta_0 $和$ \beta_1 $值，可以通过<code>summary()</code>总结参数的基本信息</p><ul><li><p>此表包含了模型的诊断信息，例如参数的均值、标准差和有效样本大小（Bulk_ESS 和 Tail_ESS）。</p></li><li><p>还提供了每个参数的 95% 最高密度区间（HDI），用于展示参数的不确定性范围。</p></li></ul><p></p>

In [31]:
summary(linear_model)

 Family: gaussian 
  Links: mu = identity 
Formula: RT_sec ~ Label 
   Data: df (Number of observations: 105) 
  Draws: 4 chains, each with iter = 6000; warmup = 1000; thin = 1;
         total post-warmup draws = 20000

Regression Coefficients:
          Estimate Est.Error l-95% CI u-95% CI Rhat Bulk_ESS Tail_ESS
Intercept     0.71      0.04     0.64     0.78 1.00    20510    14172
Label         0.15      0.05     0.06     0.24 1.00    19820    14815

Further Distributional Parameters:
      Estimate Est.Error l-95% CI u-95% CI Rhat Bulk_ESS Tail_ESS
sigma     0.23      0.02     0.20     0.27 1.00    16627    14448

Draws were sampled using sampling(NUTS). For each parameter, Bulk_ESS
and Tail_ESS are effective sample size measures, and Rhat is the potential
scale reduction factor on split chains (at convergence, Rhat = 1).

* 我们可以使用均值来理解生成的后验分布，通过上表我们知道  

	*  $\beta_0$ 表示 self 条件下的基准反应时间约为 0.796 秒。  

	*  $\beta_1$ 表示 self 和 other 条件下的反应时间差异非常小，几乎可以忽略不计。  

* 注意：尽管表中显示了参数的均值，但这些均值只是后验分布的一个概要信息。  
	*  我们还可以从 HDI 和标准差中观察到后验分布的广泛性，反映了模型的内在不确定性。  
	*  因此，仅使用均值生成的回归线并不足以充分展示后验分布的复杂性和不确定性。

上节课我们学习了使用 HDI + ROPE 进行检验。在这里我们假设 ($\beta_1$) 的值在 $[-0.05, 0.05]$ 范围内可以视为实用等效，  

即如果$\beta_1$落在这个范围内，说明 self 和 other 条件之间的反应时间差异可以忽略不计，从而在实践上认为两者无显著差异。  

1. **ROPE 区间**：我们设定 $[-0.05, 0.05]$ 为 ROPE 区间，表示 self 和 other 条件下的反应时间差异在此范围内被视为无显著差异。该范围表示了对“等效零效应”的假设，即认为微小的差异在实践中可以忽略。  

2. **HDI (Highest Density Interval)**：后验分布的 95% 最高密度区间（HDI）显示了 $\beta_1$ 的不确定性范围，帮助我们了解后验分布中最可信的值区域。  

3. **结果解读**：  
   - 如果 $\beta_1$ 的后验分布大部分位于 ROPE 区间内，我们可以认为 self 和 other 条件下的反应时间差异在实用上无显著意义，即这两种条件在反应时间上几乎等同。  
   - 如果后验分布的很大一部分超出了 ROPE 区间，则表明 self 和 other 条件之间的差异在实用上具有显著性，值得进一步关注。  


In [32]:
# 1. 提取β₁的后验样本
posterior_beta1 <- as_draws_df(linear_model) %>%
  select(beta1 = b_Label) %>%
  pull(beta1)

# 2. 定义ROPE区间
rope_interval <- c(-0.05, 0.05)

# 3. 计算95% HDI
hdi <- function(x, prob = 0.95) {
  x_sorted <- sort(x)
  n <- length(x_sorted)
  window_size <- ceiling(prob * n)
  min_width <- Inf
  hdi_low <- x_sorted[1]
  hdi_high <- x_sorted[window_size]
  
  for (i in 1:(n - window_size + 1)) {
    current_low <- x_sorted[i]
    current_high <- x_sorted[i + window_size - 1]
    current_width <- current_high - current_low
    if (current_width < min_width) {
      min_width <- current_width
      hdi_low <- current_low
      hdi_high <- current_high
    }
  }
  c(hdi_low, hdi_high)
}

hdi_95 <- hdi(posterior_beta1, prob = 0.95)

# 4. 计算ROPE内的后验概率
rope_prob <- mean(posterior_beta1 >= rope_interval[1] & posterior_beta1 <= rope_interval[2]) * 100

# 5. 绘制后验分布（去除黑色均值线）
density_data <- density(posterior_beta1)
density_df <- data.frame(x = density_data$x, y = density_data$y)

ggplot(density_df, aes(x = x, y = y)) +
  # 后验分布填充（浅蓝色）
  geom_area(fill = "#3498db", alpha = 0.3) +
  # 95% HDI区间（深蓝色竖线）
  geom_vline(xintercept = hdi_95, color = "#2980b9", linetype = "solid", linewidth = 0.7) +
  # ROPE区间（灰色虚线边框）
  annotate(
    "rect",
    xmin = rope_interval[1], xmax = rope_interval[2],
    ymin = 0, ymax = Inf,
    fill = "grey80", alpha = 0.5,
    color = "grey50", linetype = "dashed", linewidth = 0.5
  ) +
  # （已去除：后验均值黑色竖线）
  # 标注ROPE内比例
  annotate(
    "text",
    x = mean(posterior_beta1), y = max(density_df$y) * 0.9,
    label = paste0("ROPE: ", round(rope_prob, 1), "%"),
    color = "black", size = 4
  ) +
  # 标注95% HDI范围
  annotate(
    "text",
    x = hdi_95[2], y = max(density_df$y) * 0.8,
    label = paste0("95% HDI: [", round(hdi_95[1], 3), ", ", round(hdi_95[2], 3), "]"),
    color = "#2980b9", size = 4, hjust = 1
  ) +
  # 坐标轴与标题
  xlab(expression(beta[1])) +
  ylab("Density") +
  ggtitle(expression("Posterior of" ~ beta[1])) +
  # 主题设置
  theme_minimal() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 14),
    axis.text = element_text(size = 12),
    axis.title = element_text(size = 12),
    panel.grid = element_blank()
  )

Warning message:
“Dropping 'draws_df' class as required metadata was removed.”


plot without title

<p>我们可以看到 $ \beta_1 $ 的后验分布主要集中在正值区域，其均值约为 0.15。</p><p>图中的 95% 最高密度区间（HDI）范围为 $ [0.059, 0.24] $，且大部分后验分布落在 ROPE 区间 $ [-0.05, 0.05] $ 之外，只有 1.6% 的后验分布位于 ROPE 区间内。</p><p>这表明 self 和 other 条件下的反应时间差异在实践上具有显著性，即 $ \beta_1 $ 的值足够大，可以排除两者在反应时间上的实用等效性。因此，self 和 other 条件之间的差异值得关注。</p>

<p>总结<br>- 本节课通过一个简单的线性回归示例，展示了如何使用 <code>brms</code> 构建贝叶斯模型，并结合之前的内容对模型结果进行深入分析。<br>- 我们特别关注了先验分布的设定和后验预测检查（PPC）的重要性，通过对比真实数据与预测分布，评估模型的合理性和对新数据的预测能力。<br>- 此外，我们全程基于 <code>brms</code> 简化线性模型的定义、拟合与后验分析流程——无需手动编写复杂的采样逻辑，仅通过简洁的公式和参数设置即可完成建模，让贝叶斯分析的落地更加便捷。<br>- 最后，我们强调了贝叶斯建模的关键步骤：从先验设定、模型拟合（依赖MCMC方法近似后验分布），到后验推断（如HDI区间、ROPE检验）与结果解释，完整覆盖了贝叶斯线性回归的核心流程，也进一步明确了MCMC方法在获取参数后验分布中的核心作用。</p><p></p><img src="https://cdn.kesci.com/upload/smkhdwv5zt.png?imageView2/0/w/720" alt="Image Name"><p></p>

<h2>补充材料：为什么使用MCMC是必要的</h2><blockquote><p>我们都知道当后验分布的计算过于复杂时，我们应该选用MCMC来近似后验分布</p></blockquote><blockquote><p>但是在这里后验分布究竟有多复杂呢，这里提供了直接的计算(or提供一些复杂的公式让人知难而退)：</p></blockquote><ol><li><p>该线性模型存在三个参数值$ (\beta_0, \beta_1, \sigma) $</p><ul><li><p>那么先验概率则为三者pdf的乘积：</p><p>$$ f(\beta_0, \beta_1, \sigma) = f(\beta_0) f(\beta_1) f(\sigma) $$</p></li></ul></li><li><p>观测到的数据可以用$ \vec{y} = (y_1,y_2,...,y_{n}) $来表示</p><ul><li><p>那么似然函数可以表示为：</p><p>$$ L(\beta_0, \beta_1, \sigma | \vec{y}) = f(\vec{y}|\beta_0, \beta_1, \sigma) = \prod_{i=1}^{n}f(y_i|\beta_0, \beta_1, \sigma) $$</p></li></ul></li><li><p>后验分布则可以表示为：</p><p>$$ \begin{split} f(\beta_0,\beta_1,\sigma \; | \; \vec{y}) &amp; = \frac{\text{prior} \cdot \text{likelihood}}{ \int \text{prior} \cdot \text{likelihood}} \\ &amp; = \frac{f(\beta_0) f(\beta_1) f(\sigma) \cdot \left[\prod_{i=1}^{n}f(y_i|\beta_0, \beta_1, \sigma) \right]} {\int\int\int f(\beta_0) f(\beta_1) f(\sigma) \cdot \left[\prod_{i=1}^{n}f(y_i|\beta_0, \beta_1, \sigma) \right] d\beta_0 d\beta_1 d\sigma} \\ \end{split} $$</p></li></ol><p></p>